In [2]:
import os
os.environ['XLA_PYTHON_CLIENT_MEM_FRACTION'] = '.99'

from euclidean_fast_attention.utils.npz_trainer import NpzTrainer
from euclidean_fast_attention.model import EnergyModel
from pathlib import Path
from orbax import checkpoint

import jax
import jax.numpy as jnp
import numpy as np
import optax

D2N = {10: 39, 15: 132, 20: 314, 25: 613, 30: 1060}
jax.local_devices()

[CudaDevice(id=0)]

In [ ]:

data_folder = Path(f'../datasets').resolve()

batch_size = 4

# IO.
save_folder = '../runs/NaCl/'

# Data. Download the other data from zenodo (datasets/NaCl/sphere/) and put to ../datasets to run training for multiple D > 10 in a loop.
Ds = [10]

# Model.
T = 1
model = 'efa' # other option is 'local'

# Training.
num_epochs = 100

for D in Ds:
    num_atoms = D2N[D]

    lambda_f = 10.0
    lambda_e = 1.0

    data_dir = data_folder / f'N{num_atoms}_D{D}.npz'
    data = np.load(data_dir)

    forces_weight = (lambda_f / data['forces'].var()).item()
    energy_weight = (lambda_e / data['energy'].var()).item()

    print(f'{energy_weight=} and {forces_weight=}')

    trainer = NpzTrainer(
        data_dir,
        num_train=1000,
        num_valid=200,
        max_num_nodes=batch_size*num_atoms+1,
        max_num_edges=batch_size*num_atoms*34+1,
        max_num_graphs=batch_size+1,
        num_epochs=num_epochs,
        energy_unit=1.,
        length_unit=1.,
        save_interval_steps=250,
        log_interval_steps=1_000_000,
        pbc_bool=False,
        use_wandb=False,
        forces_weight=forces_weight,
        energy_weight=energy_weight,
        subtract_energy_mean=False,
        rotation_augmentation_bool=True
    )
    
    base_model = EnergyModel(
        cutoff=5.0,
        num_features=32,
        num_layers=T,
        mp_max_degree=0,
        mp_block_post_mlp_bool=False,
        efa_block_post_mlp_bool=False,
        num_post_residual_mlps=1,
        era_activation_fn=lambda u: u,
        era_qk_num_features=128,
        era_v_num_features=8,
        era_use_in_iterations=list(range(T)) if model == 'efa' else [],
        era_max_degree=0,
        era_max_length=30.0,
        era_lebedev_num=194,
        era_max_frequency=4*np.pi,
        era_frequencies_trainable=False,
        pbc_bool=False
    )

    ckpt_dir = Path(f'{save_folder}/NaCl_toy_N{num_atoms}_D{D}_{model}_T{T}').resolve()
    trainer.run_training(
        ckpt_dir=ckpt_dir,
        model=base_model,
        optimizer=optax.adamw(1e-3),
    )

energy_weight=0.0008866098360158503 and forces_weight=1.59859299659729


2025-09-16 11:19:42.655567: W external/xla/xla/service/gpu/autotuning/dot_search_space.cc:200] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs?Working around this by using the full hints set instead.
2025-09-16 11:19:48.337407: W external/xla/xla/service/gpu/autotuning/dot_search_space.cc:200] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs?Working around this by using the full hints set instead.
2025-09-16 11:19:50.735419: W external/xla/xla/service/gpu/autotuning/dot_search_space.cc:200] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs?Working around this by using the full hints set instead.


Number of parameters: 16096


Do not subtract energy mean.


2025-09-16 11:20:00.562562: W external/xla/xla/service/gpu/autotuning/dot_search_space.cc:200] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs?Working around this by using the full hints set instead.
2025-09-16 11:20:00.562597: W external/xla/xla/service/gpu/autotuning/dot_search_space.cc:200] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs?Working around this by using the full hints set instead.
2025-09-16 11:20:00.562606: W external/xla/xla/service/gpu/autotuning/dot_search_space.cc:200] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs?Working around this by using the full hints set instead.
2025-09-16 11:20:00.562614: W external/xla/xla/service/gpu/au

Eval metrics:  {'eval_loss': 8.776769638061523, 'eval_energy_mse': 786.8080444335938, 'eval_forces_mse': 5.053929805755615}
Eval metrics:  {'eval_loss': 4.352573871612549, 'eval_energy_mse': 180.43960571289062, 'eval_forces_mse': 2.6226775646209717}
Eval metrics:  {'eval_loss': 1.1347808837890625, 'eval_energy_mse': 109.02281951904297, 'eval_forces_mse': 0.6493961215019226}
Eval metrics:  {'eval_loss': 0.07630465924739838, 'eval_energy_mse': 3.428093433380127, 'eval_forces_mse': 0.0458311066031456}
Eval metrics:  {'eval_loss': 0.06484787911176682, 'eval_energy_mse': 1.623031735420227, 'eval_forces_mse': 0.039665430784225464}
Eval metrics:  {'eval_loss': 0.07116183638572693, 'eval_energy_mse': 12.184523582458496, 'eval_forces_mse': 0.03775752708315849}
Eval metrics:  {'eval_loss': 0.0650104284286499, 'eval_energy_mse': 6.970734596252441, 'eval_forces_mse': 0.03680117428302765}
Eval metrics:  {'eval_loss': 0.05803132802248001, 'eval_energy_mse': 0.6567111015319824, 'eval_forces_mse': 0.0

In [ ]:
# Select a model for evaluation. Here we chose the D = 10 model from above.
model = 'efa'
batch_size = 4
T = 1
D = 10
num_atoms = D2N[D]

data_dir = data_folder / f'N{num_atoms}_D{D}.npz'
data = np.load(data_dir)

trainer = NpzTrainer(
    data_dir,
    num_train=1000,
    num_valid=200,
    max_num_nodes=batch_size*num_atoms+1,
    max_num_edges=batch_size*num_atoms*35+1,
    max_num_graphs=batch_size+1,
    num_epochs=100,
    energy_unit=1.,
    length_unit=1.,
    save_interval_steps=250,
    log_interval_steps=1_000_000,
    pbc_bool=False,
    use_wandb=False,
    forces_weight=1.0,
    energy_weight=1.0,
    subtract_energy_mean=False
)

base_model = EnergyModel(
    cutoff=5.0,
    num_features=32,
    num_layers=T,
    mp_max_degree=0,
    mp_block_post_mlp_bool=False,
    efa_block_post_mlp_bool=False,
    num_post_residual_mlps=1,
    era_activation_fn=lambda u: u,
    era_qk_num_features=128,
    era_v_num_features=8,
    era_use_in_iterations=list(range(T)) if model == 'efa' else [],
    era_max_degree=0,
    era_max_length=30.0,
    era_lebedev_num=194,
    era_max_frequency=4*np.pi,
    era_frequencies_trainable=False,
    pbc_bool=False
)

ckpt_dir = Path(f'{save_folder}/NaCl_toy_N{num_atoms}_D{D}_{model}_T{T}').resolve()
loaded_mngr = checkpoint.CheckpointManager(
        ckpt_dir,
        {
            'params': checkpoint.PyTreeCheckpointer(),
            'opt_state': checkpoint.PyTreeCheckpointer(),
        },
        options=checkpoint.CheckpointManagerOptions(step_prefix='ckpt'),
    )
mgr_state = loaded_mngr.restore(loaded_mngr.latest_step())
params = mgr_state.get('params')

metrics, predictions = trainer.run_testing(params, base_model, num_test=200, collect_predictions=True)
energy_predictions, forces_predictions, energy_gt, forces_gt, graphs = predictions
energy_predictions_np = jnp.concatenate(energy_predictions).reshape(-1)
energy_gt_np = jnp.concatenate(energy_gt).reshape(-1)
forces_predictions_np = jnp.concatenate(forces_predictions)
forces_gt_np = jnp.concatenate(forces_gt)

# Uncomment to save the results to .npz

# save_filename = f'{save_folder}/NaCl_toy_N{num_atoms}_D{D}_{model}_T{T}'
# np.savez(
#     save_filename, 
#     energy_predictions=energy_predictions_np,
#     energy_gt=energy_gt_np,
#     forces_predictions=forces_predictions_np,
#     forces_gt=forces_gt_np,
# )

/usr/local/lib/python3.12/site-packages/orbax/checkpoint/_src/serialization/type_handlers.py:1251: UserWarning: Sharding info not provided when restoring. Populating sharding info from sharding file. Please note restoration time will be slightly increased due to reading from file. Note also that this option is unsafe when restoring on a different topology than the checkpoint was saved with.
  warnings.warn(
2025-09-16 11:25:45.031776: W external/xla/xla/service/gpu/autotuning/dot_search_space.cc:200] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs?Working around this by using the full hints set instead.
2025-09-16 11:25:45.031801: W external/xla/xla/service/gpu/autotuning/dot_search_space.cc:200] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs?Working around this 

Stop testing after n=49 batches since 200 test structures have been reached.
Metrics are collected from 200 test structures.


In [ ]:
# We substract potential global shifts in the energy prediction.
bias = np.mean(energy_gt_np - energy_predictions_np)
rmse_e = np.sqrt(np.mean(np.square(energy_gt_np - energy_predictions_np - bias))) / D2N[D]

# Compare to Fig.3C, first purple bar for D=10.
print(f'Energy RMSE: {float(rmse_e):.3f} eV/atom')

Energy RMSE: 0.018 eV/atom
